In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

# Load main features
df = pd.read_csv("player_features_final.csv")   # change filename if different
# Load LSTM predictions (one-step or aggregated)
# If you have encoder_decoder_predictions.csv or lstm_univariate_predictions.csv:
try:
    lstm_preds = pd.read_csv("encoder_decoder_predictions.csv")   # or appropriate file
    print("Loaded LSTM preds:", lstm_preds.shape)
except FileNotFoundError:
    # If not present, try other names or compute a quick LSTM feature: last predicted value saved earlier
    lstm_preds = None
    print("LSTM preds file not found; ensure predictions CSVs exist.")

# Example: merge LSTM predictions back to main df. This depends on how you saved them:
# If you saved predictions earlier aligned with X_test indexes, make a feature DataFrame with 'sample_id' keys.
# For this example we'll create a simple feature: LSTM_last_pred by taking 'pred_t+1' from encoder_decoder_predictions.csv
if lstm_preds is not None and 'pred_t+1' in lstm_preds.columns:
    df_lstm_feat = lstm_preds[['pred_t+1']].rename(columns={'pred_t+1':'lstm_pred_next'})
    # if alignment is required, ensure indexes match; here we assume order aligns or add an index column when saving earlier.
    # Append to df by index for demo:
    df = df.reset_index(drop=True)
    df_lstm_feat = df_lstm_feat.reset_index(drop=True)
    df = pd.concat([df, df_lstm_feat], axis=1)
else:
    # fallback: create lstm_pred_next column as NaN (XGBoost can handle or fill)
    df['lstm_pred_next'] = np.nan

# Quick fill/cleanup
df['market_value_in_eur'] = pd.to_numeric(df['market_value_in_eur'], errors='coerce').fillna(0.0)
# Drop rows without target if needed
df = df.dropna(subset=['market_value_in_eur']).reset_index(drop=True)

# If you used synthetic series, you may have duplicated samples; ensure one row per sample for tree models.
print("Final training rows:", len(df))

# Split (time-aware not necessary for tree if samples are independent; but prefer train_test_split with shuffle)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)
print("train/val/test:", len(train_df), len(val_df), len(test_df))


Loaded LSTM preds: (24707, 3)
Final training rows: 24707
train/val/test: 17788 1977 4942


In [2]:
from sklearn.preprocessing import LabelEncoder

def prepare_features(df_in):
    df = df_in.copy()
    # numeric features (example)
    num_feats = ['height_in_cm','market_value_in_eur','highest_market_value_in_eur']
    for c in num_feats:
        if c not in df.columns:
            df[c] = 0.0
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)
    # use LSTM prediction as feature (if available)
    if 'lstm_pred_next' not in df.columns:
        df['lstm_pred_next'] = 0.0
    df['lstm_pred_next'] = pd.to_numeric(df['lstm_pred_next'], errors='coerce').fillna(0.0)

    # sample engineered features
    df['value_to_peak_ratio'] = df['market_value_in_eur'] / (df['highest_market_value_in_eur'].replace(0,np.nan))
    df['value_to_peak_ratio'] = df['value_to_peak_ratio'].fillna(0.0)

    # label encode small-cardinality categorical features
    cat_cols = []
    for c in ['position','current_club_name','country_of_citizenship']:
        if c in df.columns:
            cat_cols.append(c)
            le = LabelEncoder()
            df[c+'_le'] = le.fit_transform(df[c].astype(str))
    # build final feature list
    feature_cols = ['market_value_in_eur','highest_market_value_in_eur','value_to_peak_ratio','lstm_pred_next']
    feature_cols += [c+'_le' for c in cat_cols]
    # also include sentiment if present
    if 'sentiment_score' in df.columns:
        df['sentiment_score'] = pd.to_numeric(df['sentiment_score'], errors='coerce').fillna(0.0)
        feature_cols.append('sentiment_score')

    return df, feature_cols

train_df_f, feature_cols = prepare_features(train_df)
val_df_f, _ = prepare_features(val_df)
test_df_f, _ = prepare_features(test_df)

X_train = train_df_f[feature_cols]
y_train = train_df_f['market_value_in_eur']
X_val = val_df_f[feature_cols]; y_val = val_df_f['market_value_in_eur']
X_test = test_df_f[feature_cols]; y_test = test_df_f['market_value_in_eur']

print("Features used:", feature_cols)


Features used: ['market_value_in_eur', 'highest_market_value_in_eur', 'value_to_peak_ratio', 'lstm_pred_next', 'position_le', 'current_club_name_le', 'country_of_citizenship_le', 'sentiment_score']


In [3]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import joblib

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    "objective":"reg:squarederror",
    "eval_metric":"rmse",
    "eta":0.05,
    "max_depth":6,
    "subsample":0.8,
    "colsample_bytree":0.8,
    "seed":42
}
evallist = [(dtrain,"train"), (dval,"val")]
bst = xgb.train(params, dtrain, num_boost_round=1000, evals=evallist, early_stopping_rounds=30, verbose_eval=50)

# Predict and metrics
y_pred_xgb = bst.predict(dtest)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
print("XGBoost RMSE:", rmse_xgb, "MAE:", mae_xgb)

# save model
bst.save_model("xgboost_week6.model")
joblib.dump(feature_cols, "week6_feature_cols.pkl")


[0]	train-rmse:0.86260	val-rmse:0.84855
[50]	train-rmse:0.07310	val-rmse:0.07179
[100]	train-rmse:0.01020	val-rmse:0.01360
[150]	train-rmse:0.00502	val-rmse:0.01030
[200]	train-rmse:0.00401	val-rmse:0.00968
[250]	train-rmse:0.00340	val-rmse:0.00934
[300]	train-rmse:0.00294	val-rmse:0.00917
[350]	train-rmse:0.00256	val-rmse:0.00886
[400]	train-rmse:0.00225	val-rmse:0.00874
[450]	train-rmse:0.00203	val-rmse:0.00868
[500]	train-rmse:0.00184	val-rmse:0.00863
[550]	train-rmse:0.00168	val-rmse:0.00857
[600]	train-rmse:0.00154	val-rmse:0.00855
[650]	train-rmse:0.00142	val-rmse:0.00852
[700]	train-rmse:0.00130	val-rmse:0.00848
[750]	train-rmse:0.00121	val-rmse:0.00847
[800]	train-rmse:0.00112	val-rmse:0.00845
[850]	train-rmse:0.00105	val-rmse:0.00844
[900]	train-rmse:0.00099	val-rmse:0.00844
[917]	train-rmse:0.00097	val-rmse:0.00844
XGBoost RMSE: 0.008211410864076682 MAE: 0.004119907855032234


C:\Users\saura\AppData\Local\Temp\ipykernel_3496\1673364535.py:29: UserWarning: [19:02:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  bst.save_model("xgboost_week6.model")


['week6_feature_cols.pkl']

In [5]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

params_lgb = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'seed': 42
}

# ---------------------------
# ✅ Correct way: use callbacks
# ---------------------------
callbacks = [
    lgb.early_stopping(stopping_rounds=30),
    lgb.log_evaluation(period=50)
]

gbm = lgb.train(
    params_lgb,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=callbacks
)

# Predict
y_pred_lgb = gbm.predict(X_test, num_iteration=gbm.best_iteration)

# Metrics
rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
mae_lgb = mean_absolute_error(y_test, y_pred_lgb)

print("LightGBM RMSE:", rmse_lgb, "MAE:", mae_lgb)

# Save model
gbm.save_model("lightgbm_week6.txt")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000755 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1015
[LightGBM] [Info] Number of data points in the train set: 17788, number of used features: 8
[LightGBM] [Info] Start training from score -0.000866
Training until validation scores don't improve for 30 rounds
[50]	training's rmse: 0.0897693	valid_1's rmse: 0.0998705
[100]	training's rmse: 0.0179396	valid_1's rmse: 0.0432157
[150]	training's rmse: 0.0119359	valid_1's rmse: 0.0403817
[200]	training's rmse: 0.00953185	valid_1's rmse: 0.0396851
[250]	training's rmse: 0.00833459	valid_1's rmse: 0.0391897
[300]	training's rmse: 0.00744225	valid_1's rmse: 0.0387901
[350]	training's rmse: 0.00685001	valid_1's rmse: 0.0385888
[400]	training's rmse: 0.00631893	valid_1's rmse: 0.0384897
[450]	training's rmse: 0.00589306	valid_1's rmse: 0.03838

In [6]:
from sklearn.linear_model import Ridge
# get validation predictions to train meta-learner
val_pred_xgb = bst.predict(xgb.DMatrix(X_val))
val_pred_lgb = gbm.predict(X_val, num_iteration=gbm.best_iteration)

# For LSTM pred, use df val's 'lstm_pred_next' (or predictions aligned previously)
val_lstm = val_df_f['lstm_pred_next'].values if 'lstm_pred_next' in val_df_f.columns else np.zeros(len(X_val))

# Build meta training set (for stacking)
meta_X_train = np.vstack([val_pred_xgb, val_pred_lgb, val_lstm]).T
meta_y_train = y_val.values

# For test
test_pred_xgb = y_pred_xgb
test_pred_lgb = y_pred_lgb
test_lstm = test_df_f['lstm_pred_next'].values if 'lstm_pred_next' in test_df_f.columns else np.zeros(len(X_test))
meta_X_test = np.vstack([test_pred_xgb, test_pred_lgb, test_lstm]).T

# Train meta model
meta = Ridge(alpha=1.0)
meta.fit(meta_X_train, meta_y_train)

meta_pred = meta.predict(meta_X_test)
rmse_meta = np.sqrt(mean_squared_error(y_test, meta_pred))
mae_meta = mean_absolute_error(y_test, meta_pred)
print("Stacked Ensemble RMSE:", rmse_meta, "MAE:", mae_meta)

# Save meta model
joblib.dump(meta, "week6_meta_ridge.pkl")


Stacked Ensemble RMSE: 0.012139671737699732 MAE: 0.00900258668554606


['week6_meta_ridge.pkl']

In [10]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# === Load training, validation, and test sets (you created these earlier) ===
# Make sure X_train, y_train, X_val, y_val, X_test, y_test exist in your notebook.

model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# Evaluate
preds = model_xgb.predict(X_test)
rmse = mean_squared_error(y_test, preds) ** 0.5
print("XGBoost RMSE:", rmse)



XGBoost RMSE: 0.008955442401980195


In [11]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.savefig("week6_shap_summary.png")
plt.close()

print("SHAP summary saved as week6_shap_summary.png")


SHAP summary saved as week6_shap_summary.png


In [12]:
# save predictions CSV
out = pd.DataFrame({
    'actual': y_test,
    'xgb_pred': y_pred_xgb,
    'lgb_pred': y_pred_lgb,
    'meta_pred': meta_pred
})
out.to_csv("week6_preds_comparison.csv", index=False)
print("Saved week6_preds_comparison.csv")


Saved week6_preds_comparison.csv
